# C4/C5: HAVC lúc inference (Qwen2.5-VL-3B) trên tập dev

C4 dùng expert head chọn bằng chữ tiếng Anh, C5 dùng expert head chọn bằng chữ tiếng Việt có dấu (từ pilot chẩn đoán).
Hai nhánh dùng chung một lần forward + backward cho mỗi câu, cùng câu hỏi, cùng prompt; chỉ khác tập head.

Mỗi câu: lọc expert head theo entropy không gian, gộp với độ nhạy gradient, lấy top-K, gộp bản đồ, lấy hộp như ViCrop,
rồi trả lời với hai chế độ đưa ảnh (`crop`: chỉ ảnh crop như C3 hiện tại; `orig_crop`: ảnh gốc + ảnh crop như paper HAVC)
và bốn kích thước hộp giống lượt quét C3.

Chỉ chạy **dev**. Tập test để dành chạy một lần sau khi chốt cấu hình.

**Lần chạy hiện tại là chạy thử pipeline:** dev cũ của C3 có 13/21 câu thuộc tài liệu đã bị loại (An Bằng), bỏ đi
còn 8 câu của một tài liệu, chỉ đủ kiểm tra VRAM, thời gian và số head qua ngưỡng entropy, không đủ chốt cấu hình.
Khi đã chia dev/test theo tài liệu, đổi `DEV_FILE` ở ô expert head rồi chạy lại.

**Cần:** dataset `da2-dev` (bản đã dùng cho C3; notebook tự ghi đè code mới, không cần tải lại), GPU T4 x2, Internet On.

**Thời gian ước tính:** khoảng 1 phút/câu (8 câu khoảng 10 phút), chưa tính tải model.

**Kết quả:** tải `/kaggle/working/havc_dev_results.zip`, giải nén vào gốc repo.

In [ ]:
!pip install -q transformers accelerate qwen-vl-utils pillow numpy scipy
import os, sys, shutil, glob

hits = glob.glob('/kaggle/input/**/vicrop_relatt.py', recursive=True)
assert hits, 'Không thấy vicrop_relatt.py: kiểm tra đã Add Input dataset da2-dev chưa'
src_root = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))
print('Dataset root:', src_root)
shutil.copytree(src_root, '/kaggle/working/da2', dirs_exist_ok=True)
os.chdir('/kaggle/working/da2'); sys.path.insert(0, '.')

import torch
print(torch.cuda.device_count(), 'GPU:', torch.cuda.get_device_name(0))

In [ ]:
%%writefile src/crop/havc.py
"""HAVC lúc inference (nhánh C4, C5) cho Qwen2.5-VL: lọc expert head theo entropy không gian và độ nhạy gradient,
gộp K head thành bản đồ hướng dẫn, rồi lấy hộp crop như ViCrop.

Theo Xie và cs., arXiv:2601.22483, mục 2.2 (đã đọc toàn văn phần phương pháp):
  - Entropy không gian (Eq. 5): bản đồ attention của head chuẩn hóa về [0,1], nhị phân bằng Otsu, tách C_h thành
    phần liên thông; E_h = min(λ_c (C_h - 1) + λ_d · d̄_h / d_max, 1), d̄_h = khoảng cách trung bình giữa các tâm
    thành phần, d_max = đường chéo bản đồ. Chỉ giữ head có E_h < 0,3.
  - Độ nhạy gradient (Eq. 6-7): S_h^sens = ∂ log p(y*) / ∂ a_h, G_h = <a_h, max(0, S_h^sens)>.
  - Gộp (Eq. 8): S_h = α N(1 - E_h) + (1 - α) N(G_h), N = min-max trên các head; lấy top-K theo S_h.
  - Trọng số (Eq. 9-10): w_h = softmax(S_h / τ) trên K head; M_final = Σ w_h A_h. Hộp lấy từ M_final theo ViCrop.
  Siêu tham số theo paper: K = 8, α = 0,4, τ = 0,1, ngưỡng entropy 0,3 (ngưỡng 0,5 của bước chọn expert head nằm
  ở src/diagnosis/analyze_heads.py).

Chỗ paper không nói rõ, tự chọn (phải nêu trong báo cáo):
  - λ_c, λ_d: paper ghi "theo [8]" (Kang và cs., CVPR 2025) nhưng [8] dùng công thức entropy Shannon khác, không
    có λ. Tự đặt LAMBDA_C, LAMBDA_D bên dưới; đã lưu bản đồ từng head để xem lại trên dev.
  - Liên thông 8 hướng (theo [8]); paper HAVC không nói.
  - a_h = attention của token đầu vào cuối cùng (vị trí sinh token đầu tiên) lên các token ảnh, như ViCrop;
    y* = token đầu tiên mô hình đoán (argmax). Paper chỉ nói "truy vấn giải mã hiện tại" và "token dự đoán".
  - Bản đồ thô, không chia cho attention của prompt chung như rel_att của ViCrop (paper không nhắc).
  - Entropy và gradient chỉ tính trên expert head của bước chẩn đoán.
  - Còn ít hơn K head qua ngưỡng entropy thì dùng hết số còn lại; không còn head nào thì lấy K head entropy thấp
    nhất và đánh dấu fallback.
  - Đạo hàm nhân GRAD_SCALE trước backward để tránh underflow ở fp16, rồi chia lại; N(G_h) không đổi.
"""

import numpy as np

K = 8
ALPHA = 0.4
TAU = 0.1
ENTROPY_THRESH = 0.3
LAMBDA_C = 0.1  # tự chọn, paper không cho giá trị
LAMBDA_D = 0.5  # tự chọn, paper không cho giá trị
GRAD_SCALE = 1024.0


def otsu_threshold(x: np.ndarray, bins: int = 256) -> float:
    """Ngưỡng Otsu cho giá trị trong [0, 1]: cực đại phương sai giữa hai lớp."""
    hist, edges = np.histogram(x.ravel(), bins=bins, range=(0.0, 1.0))
    p = hist.astype(float) / max(1, hist.sum())
    mid = (edges[:-1] + edges[1:]) / 2
    w0 = np.cumsum(p)
    mu = np.cumsum(p * mid)
    w1 = 1.0 - w0
    with np.errstate(divide="ignore", invalid="ignore"):
        between = (mu[-1] * w0 - mu) ** 2 / (w0 * w1)
    between[~np.isfinite(between)] = -1
    return float(edges[int(np.argmax(between)) + 1])


def spatial_entropy(att_map: np.ndarray, lam_c: float = LAMBDA_C, lam_d: float = LAMBDA_D):
    """Trả về (E_h, C_h, d̄_h / d_max) của một bản đồ attention (H_grid, W_grid)."""
    from scipy import ndimage
    m = att_map.astype(float)
    m = (m - m.min()) / (m.max() - m.min() + 1e-12)
    labels, c = ndimage.label(m > otsu_threshold(m), structure=np.ones((3, 3)))
    if c < 2:
        return 0.0, int(c), 0.0
    cent = np.array(ndimage.center_of_mass(np.ones_like(m), labels, range(1, c + 1)))
    dist = np.sqrt(((cent[:, None] - cent[None]) ** 2).sum(-1))
    d_rel = float(dist[np.triu_indices(c, 1)].mean() / np.hypot(*m.shape))
    return float(min(lam_c * (c - 1) + lam_d * d_rel, 1.0)), int(c), d_rel


def _minmax(x: np.ndarray) -> np.ndarray:
    return (x - x.min()) / (x.max() - x.min() + 1e-12)


def head_maps_and_grads(model, processor, image, prompt: str, heads):
    """Một lần forward + backward. Trả về (maps {(l, h): (H_grid, W_grid)}, grads {(l, h): G_h}, y*).

    Cần attn_implementation='eager' và tham số model đã requires_grad_(False) (chỉ cần gradient theo attention).
    Gradient chạy được nhờ hook đặt requires_grad cho embedding đầu vào; phần vision encoder không giữ đồ thị.
    """
    import torch
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = processor(text=[text], images=[image], return_tensors="pt").to(model.device)
    vis = (inp["input_ids"][0] == model.config.image_token_id).nonzero().flatten()
    t, gh2, gw2 = inp["image_grid_thw"][0].tolist()
    grid = (gh2 // 2, gw2 // 2)
    assert t == 1 and len(vis) == grid[0] * grid[1], (t, grid, len(vis))

    layers = sorted({l for l, _ in heads})
    hook = model.get_input_embeddings().register_forward_hook(lambda m, i, o: o.requires_grad_(True))
    try:
        with torch.enable_grad():
            out = model(**inp, output_attentions=True)
            logp = out.logits[0, -1].float().log_softmax(-1)
            y = int(logp.argmax())
            atts = [out.attentions[l] for l in layers]
            grads = torch.autograd.grad(logp[y] * GRAD_SCALE, atts)
    finally:
        hook.remove()

    maps, g_h = {}, {}
    for l, a, g in zip(layers, atts, grads):
        v = vis.to(a.device)
        a_q = a[0, :, -1, v].detach().float()  # (heads, n_vis)
        g_q = g[0, :, -1, v].float() / GRAD_SCALE
        sens = (a_q * g_q.clamp(min=0)).sum(-1).cpu().numpy()
        a_q = a_q.cpu().numpy().reshape(-1, *grid)
        for (ll, h) in heads:
            if ll == l:
                maps[(l, h)], g_h[(l, h)] = a_q[h], float(sens[h])
    del out, atts, grads
    return maps, g_h, y


def guidance_map(maps, grads, heads, k=K, alpha=ALPHA, tau=TAU, e_thresh=ENTROPY_THRESH,
                 lam_c=LAMBDA_C, lam_d=LAMBDA_D):
    """Lọc entropy, gộp với gradient, lấy top-K, trả về (M_final, info). heads: expert head theo thứ tự chẩn đoán."""
    ent = {h: spatial_entropy(maps[h], lam_c, lam_d) for h in heads}
    kept = [h for h in heads if ent[h][0] < e_thresh]
    fallback = not kept
    if fallback:
        kept = sorted(heads, key=lambda h: ent[h][0])[:k]
    s = alpha * _minmax(np.array([1 - ent[h][0] for h in kept])) + \
        (1 - alpha) * _minmax(np.array([grads[h] for h in kept]))
    order = np.argsort(-s, kind="stable")[:k]
    top = [kept[i] for i in order]
    w = np.exp((s[order] - s[order].max()) / tau)
    w /= w.sum()
    m_final = sum(wi * maps[h] for wi, h in zip(w, top))
    info = {"n_experts": len(heads), "n_pass_entropy": 0 if fallback else len(kept), "fallback": fallback,
            "top_heads": [[int(l), int(h)] for l, h in top], "weights": [round(float(x), 4) for x in w],
            "entropy_top": [round(ent[h][0], 3) for h in top],
            "median_components": float(np.median([ent[h][1] for h in heads]))}
    return m_final, info


In [ ]:
%%writefile src/crop/vicrop_relatt.py
"""ViCrop `rel_att` cho Qwen2.5-VL (tự cài lại, nhánh C3).

Bám cơ chế của saccharomycetes/mllms_know (qwen2_5_methods.py, utils.py), nhưng
được cài lại từ phần mô tả code gốc, chưa đối chiếu từng dòng. Cần diff với
bản gốc trước khi báo cáo; mọi chỗ khác gốc phải ghi vào báo cáo.

Cơ chế:
  1. Forward 2 lần cùng ảnh: prompt câu hỏi và prompt chung.
  2. Lấy attention của token cuối tới các token ảnh ở layer ATT_LAYER, trung bình theo head.
  3. rel_att = att_câu_hỏi / att_prompt_chung (khử thiên lệch vị trí).
  4. Reshape về lưới (h/2, w/2), tìm cửa sổ có tổng attention cao nhất ở nhiều tỉ lệ.
"""

import numpy as np

ATT_LAYER = 22  # mặc định của repo gốc
RATIOS = (1, 1.2, 1.4, 1.6, 1.8, 2)
GENERAL_PROMPT = "Write a general description of the image."  # [cần đối chiếu với repo gốc]


def bbox_from_att_map(att_map: np.ndarray, image_size: tuple, bbox_size: int):
    """Chọn hộp vuông quanh vùng attention cao nhất.

    att_map: (H_grid, W_grid). image_size: (W, H) pixel. bbox_size: cạnh hộp cơ sở (pixel).
    Trả về (x1, y1, x2, y2) pixel, nằm trong ảnh.
    """
    W, H = image_size
    gh, gw = att_map.shape
    bw, bh = W / gw, H / gh  # kích thước 1 ô lưới theo pixel

    best = None  # (độ nhọn, tâm_x, tâm_y, cạnh)
    for r in RATIOS:
        side = bbox_size * r
        nx, ny = max(1, round(side / bw)), max(1, round(side / bh))
        nx, ny = min(nx, gw), min(ny, gh)

        # tổng attention của mọi cửa sổ nx x ny, qua ảnh tích phân
        ii = np.pad(att_map, ((1, 0), (1, 0))).cumsum(0).cumsum(1)
        sums = ii[ny:, nx:] - ii[:-ny, nx:] - ii[ny:, :-nx] + ii[:-ny, :-nx]
        y, x = np.unravel_index(np.argmax(sums), sums.shape)
        peak = sums[y, x]

        # độ nhọn: đỉnh so với trung bình các cửa sổ lân cận, chuẩn hóa theo diện tích
        neigh = [sums[j, i] for j in (y - 1, y, y + 1) for i in (x - 1, x, x + 1)
                 if (j, i) != (y, x) and 0 <= j < sums.shape[0] and 0 <= i < sums.shape[1]]
        sharp = (peak - (np.mean(neigh) if neigh else 0.0)) / (nx * ny)
        if best is None or sharp > best[0]:
            best = (sharp, (x + nx / 2) * bw, (y + ny / 2) * bh, side)

    _, cx, cy, side = best
    side = min(side, W, H)
    cx = min(max(cx, side / 2), W - side / 2)
    cy = min(max(cy, side / 2), H - side / 2)
    return (int(cx - side / 2), int(cy - side / 2), int(cx + side / 2), int(cy + side / 2))


def rel_att_map(model, processor, image, question_prompt: str, layer: int = ATT_LAYER):
    """Trả về rel_att (H_grid, W_grid) của một layer."""
    return rel_att_maps(model, processor, image, question_prompt, (layer,))[layer]


def rel_att_maps(model, processor, image, question_prompt: str, layers=(ATT_LAYER,)):
    """Trả về {layer: rel_att (H_grid, W_grid)}. Một lần forward cho mọi layer.

    Chạy trên GPU, cần attn_implementation='eager'.
    """
    import torch

    def run(prompt):
        msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
        text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inp = processor(text=[text], images=[image], return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model(**inp, output_attentions=True)
        ids = inp["input_ids"][0]
        tok = processor.tokenizer
        pos = (ids == tok.convert_tokens_to_ids("<|vision_start|>")).nonzero()[0].item() + 1
        end = (ids == tok.convert_tokens_to_ids("<|vision_end|>")).nonzero()[0].item()
        grid = (inp["image_grid_thw"][0, 1:] // 2).tolist()
        return {l: out.attentions[l][0, :, -1, pos:end].mean(dim=0).float().cpu().numpy().reshape(grid)
                for l in layers}

    q, g = run(question_prompt), run(GENERAL_PROMPT)
    return {l: q[l] / (g[l] + 1e-8) for l in layers}


In [ ]:
# expert head từ pilot chẩn đoán (results/*/expert_heads_config.json trong repo)
import json
from pathlib import Path
HEADS_CFG = {"C4_havc_en": {"source": "diagnosis_en.npz", "thresh": 0.5, "n_heads": 35, "heads": [[32, 7], [20, 7], [20, 1], [25, 5], [7, 3], [20, 3], [30, 10], [31, 15], [30, 11], [20, 4], [20, 5], [5, 12], [20, 2], [31, 8], [25, 4], [30, 15], [30, 3], [31, 7], [30, 14], [32, 3], [22, 15], [22, 13], [31, 12], [22, 10], [27, 2], [7, 4], [27, 6], [30, 12], [29, 3], [28, 10], [28, 13], [26, 15], [27, 0], [27, 12], [27, 7]], "score_norm": [1.0, 0.9841, 0.9666, 0.958, 0.935, 0.9231, 0.9167, 0.8794, 0.8573, 0.8567, 0.8373, 0.8366, 0.8006, 0.7742, 0.7596, 0.7519, 0.7433, 0.7291, 0.7056, 0.7029, 0.6879, 0.6747, 0.6706, 0.6354, 0.6049, 0.5804, 0.5722, 0.5683, 0.5649, 0.5643, 0.5474, 0.5421, 0.5399, 0.5255, 0.5012]}, "C5_havc_vi": {"source": "diagnosis_vi.npz", "thresh": 0.5, "n_heads": 44, "heads": [[32, 7], [20, 7], [20, 4], [32, 3], [25, 5], [30, 3], [20, 1], [31, 15], [30, 10], [30, 11], [31, 7], [20, 3], [7, 3], [20, 5], [25, 4], [30, 14], [31, 8], [31, 12], [20, 2], [30, 15], [5, 12], [29, 3], [22, 13], [27, 2], [27, 6], [29, 2], [22, 15], [28, 13], [28, 10], [29, 4], [22, 10], [29, 1], [27, 12], [29, 5], [29, 0], [26, 15], [27, 7], [26, 9], [30, 1], [26, 7], [27, 4], [27, 0], [26, 12], [27, 3]], "score_norm": [1.0, 0.9508, 0.9432, 0.9317, 0.921, 0.9187, 0.8987, 0.8985, 0.8979, 0.8959, 0.8655, 0.8398, 0.834, 0.8292, 0.819, 0.7969, 0.7586, 0.7169, 0.7076, 0.707, 0.695, 0.6896, 0.6861, 0.6759, 0.6368, 0.6309, 0.6266, 0.6064, 0.6029, 0.6028, 0.5995, 0.5885, 0.585, 0.5764, 0.5755, 0.5683, 0.5675, 0.5606, 0.5575, 0.5315, 0.5159, 0.5106, 0.5084, 0.5082]}}
for b, d in HEADS_CFG.items():
    Path(f'results/{b}').mkdir(parents=True, exist_ok=True)
    json.dump(d, open(f'results/{b}/expert_heads_config.json', 'w'), indent=1)
    print(b, d['n_heads'], 'expert head')

# tài liệu bị loại khỏi bộ dữ liệu (data/qa/excluded_documents.json): mọi nhánh bỏ qua
EXCLUDED = ["01d7b45bc59e9652f1764e6fbec2f05e", "afaa20365d8273ef0ff4601d83748253"]
# Đổi sang file dev mới khi chốt chia dev/test theo tài liệu; KHÔNG trỏ vào file test
DEV_FILE = 'data/processed/visr_bench_vi/qa_vi_haiku_dev.jsonl'

In [ ]:
import time, random, unicodedata
import numpy as np
from PIL import Image
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from src.crop import havc
from src.crop.vicrop_relatt import bbox_from_att_map

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

CFG = dict(
    model='Qwen/Qwen2.5-VL-3B-Instruct', dtype='float16', attn='eager',
    max_pixels=1024 * 28 * 28, bbox_sizes=[336, 512, 768, 1024], modes=['crop', 'orig_crop'],
    K=havc.K, alpha=havc.ALPHA, tau=havc.TAU, entropy_thresh=havc.ENTROPY_THRESH,
    lambda_c=havc.LAMBDA_C, lambda_d=havc.LAMBDA_D, grad_scale=havc.GRAD_SCALE,
    att_query='token đầu vào cuối; y* = token đầu tiên đoán ra', att_map='attention thô (không rel_att)',
    answer_suffix='\nTrả lời ngắn gọn bằng một cụm từ hoặc một câu.',
    max_new_tokens=64, split='dev', seed=SEED,
)
processor = AutoProcessor.from_pretrained(CFG['model'], max_pixels=CFG['max_pixels'])
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    CFG['model'], torch_dtype=torch.float16, device_map='auto', attn_implementation='eager').eval()
for p in model.parameters():
    p.requires_grad_(False)  # chỉ cần gradient theo attention, không theo trọng số

In [ ]:
def answer(images, question):
    # Cùng prompt sinh đáp án với C3; nhiều ảnh thì đặt các ảnh trước câu hỏi theo thứ tự
    content = [{'type': 'image'} for _ in images] + [{'type': 'text', 'text': question + CFG['answer_suffix']}]
    text = processor.apply_chat_template([{'role': 'user', 'content': content}], tokenize=False, add_generation_prompt=True)
    inp = processor(text=[text], images=list(images), return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=CFG['max_new_tokens'], do_sample=False)
    res = processor.batch_decode(out[:, inp['input_ids'].shape[1]:], skip_special_tokens=True)[0]
    return unicodedata.normalize('NFC', res.strip())

BRANCHES = list(HEADS_CFG)
HEADS = {b: [tuple(h) for h in HEADS_CFG[b]['heads']] for b in BRANCHES}
UNION = sorted(set().union(*HEADS.values()))
assert 'test' not in DEV_FILE, 'Chỉ chạy dev'
dev = [json.loads(l) for l in open(DEV_FILE, encoding='utf-8') if l.strip()]
dev = [it for it in dev if it['document_id'] not in EXCLUDED]
print(len(dev), 'câu dev sau khi bỏ tài liệu bị loại;', len({it['document_id'] for it in dev}), 'tài liệu')
OUT = {b: Path(f'results/{b}') for b in BRANCHES}
for b in BRANCHES:
    json.dump(CFG | {'branch': b, 'n_expert_heads': len(HEADS[b]), 'dev_file': DEV_FILE, 'n_questions': len(dev),
                     'excluded_documents': EXCLUDED},
              open(OUT[b] / 'dev_sweep_config.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
fout = {b: open(OUT[b] / 'dev_sweep.jsonl', 'w', encoding='utf-8') for b in BRANCHES}
store = {b: {} for b in BRANCHES}

for n, it in enumerate(dev, 1):
    img = Image.open(f"data/raw/visr_bench_vi/{it['image']}").convert('RGB')
    q = unicodedata.normalize('NFC', it['question'])
    torch.cuda.reset_peak_memory_stats(); t_q = time.time()
    full = answer([img], q)
    t0 = time.time()
    maps, grads, y = havc.head_maps_and_grads(model, processor, img, q + CFG['answer_suffix'], UNION)
    sec_havc, vram_havc = round(time.time() - t0, 2), round(torch.cuda.max_memory_allocated() / 2**30, 2)
    torch.cuda.empty_cache()
    cache, log = {}, []  # C4 và C5 ra cùng hộp thì dùng lại đáp án (giải mã tham lam nên như nhau)
    for b in BRANCHES:
        m, info = havc.guidance_map(maps, grads, HEADS[b])
        store[b][it['qid']] = (m, np.stack([maps[h] for h in HEADS[b]]), np.array([grads[h] for h in HEADS[b]]))
        log.append(f"{b[:2]} qua entropy {info['n_pass_entropy']}/{info['n_experts']}")
        for bs in CFG['bbox_sizes']:
            box = bbox_from_att_map(m, img.size, bs)
            for mode in CFG['modes']:
                if (box, mode) not in cache:
                    crop = img.crop(box)
                    cache[(box, mode)] = answer([crop] if mode == 'crop' else [img, crop], q)
                row = dict(qid=it['qid'], document_id=it['document_id'], question=q, gold=it['answer'],
                           mode=mode, bbox_size=bs, bbox=list(box), image_size=list(img.size),
                           pred_full=full, pred_crop=cache[(box, mode)], havc=info, first_token=y,
                           sec_havc=sec_havc, vram_gb_havc=vram_havc,
                           vram_gb_question=round(torch.cuda.max_memory_allocated() / 2**30, 2))
                fout[b].write(json.dumps(row, ensure_ascii=False) + '\n'); fout[b].flush()
    print(f"{n}/{len(dev)} {it['qid']} | HAVC {sec_havc}s, {vram_havc} GB | cả câu {time.time() - t_q:.1f}s | "
          + ' | '.join(log))

for b in BRANCHES:
    fout[b].close()
    qids = list(store[b])
    np.savez_compressed(OUT[b] / 'dev_head_maps.npz', qids=np.array(qids), heads=np.array(HEADS[b]),
                        **{f'{q}__final': store[b][q][0] for q in qids},
                        **{f'{q}__maps': store[b][q][1].astype(np.float16) for q in qids},
                        **{f'{q}__grads': store[b][q][2] for q in qids})
print('Xong')

In [ ]:
!cd /kaggle/working/da2 && zip -qr /kaggle/working/havc_dev_results.zip results/C4_havc_en/dev_* results/C5_havc_vi/dev_*
!ls -la /kaggle/working/havc_dev_results.zip